https://365datascience.com/tutorials/python-tutorials/pca-k-means/

In [ ]:
# Load general libraries
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.graph_objs as go
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score

## Data load

In [ ]:
nuclei = pd.read_csv('/app/analysis/watershed_nuclei.txt', sep='\t', index_col = 1)
membrane = pd.read_csv('/app/analysis/object_membrane.txt', sep='\t', index_col = 1)
cytoplasm = pd.read_csv('/app/analysis/filtered_segmented_cytoplasm.txt', sep='\t', index_col = 1)

In [ ]:
nuclei = nuclei.loc[:, ~nuclei.columns.str.contains(r'\.\d+$')]
membrane = membrane.loc[:, ~membrane.columns.str.contains(r'\.\d+$')]
cytoplasm = cytoplasm.loc[:, ~cytoplasm.columns.str.contains(r'\.\d+$')]

In [ ]:
columns_to_drop = [range(7)]  # Posiciones de las columnas a eliminar

nuclei = nuclei.drop(nuclei.columns[columns_to_drop], axis=1)
membrane = membrane.drop(membrane.columns[columns_to_drop], axis=1)
cytoplasm = cytoplasm.drop(cytoplasm.columns[columns_to_drop], axis=1)

In [ ]:
col = nuclei.pop('Children_filtered_segmented_cytoplasm_Count')
nuclei.insert(0, 'Children_filtered_segmented_cytoplasm_Count', col)
display(nuclei)

In [ ]:
display(membrane)

In [ ]:
col = cytoplasm.pop('Parent_watershed_nuclei')
cytoplasm.insert(0, 'Parent_watershed_nuclei', col)
display(cytoplasm)

In [ ]:
sns.pairplot(nuclei)

In [ ]:
# Crear el pairplot
pairplot = sns.pairplot(nuclei)

# Guardar el gráfico como un archivo SVG
plt.savefig('/app/pairplot.svg', format='svg')

# Opcional: cerrar la figura para liberar memoria
plt.close()

In [ ]:
x_label = 'AreaShape_EquivalentDiameter'
y_label = 'AreaShape_MajorAxisLength'

In [ ]:
x = nuclei['AreaShape_EquivalentDiameter']
y = nuclei['AreaShape_MajorAxisLength']  

In [ ]:
plt.figure()
plt.scatter(x, y)
plt.xlabel(x_label)
plt.ylabel(y_label)

# Añadir un título (opcional)
plt.title('Scatter Plot with Labels')

# Mostrar el gráfico
plt.show()

# Nuclei

## Standarization

In [ ]:
scaler = StandardScaler()
nuclei_std = scaler.fit_transform(nuclei)

## PCA

In [ ]:
pca = PCA()
pca.fit(nuclei_std)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
plt.figure()

plt.plot(pca.explained_variance_ratio_.cumsum(), marker = 'o', color='black')
plt.xlabel('Number of components')
plt.ylabel('Explained variance')
plt.axhline(y=0.99, color='g', linestyle='--', linewidth=0.8, label='y = 0.99')
plt.axhline(y=0.95, color='orange', linestyle='--', linewidth=0.8, label='y = 0.8')
plt.axhline(y=0.8, color='r', linestyle='--', linewidth=0.8, label='y = 0.8')
plt.xlim(0, 8)
# Añadir un título (opcional)
plt.title('Explanied variance')

# Mostrar el gráfico
plt.show()

In [ ]:
pca = PCA(n_components = 3)

In [ ]:
pca.fit(nuclei_std)

In [ ]:
pca_result = pca.transform(nuclei_std)
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2', 'PC3'])

fig = px.scatter_3d(pca_df, x='PC1', y='PC2', z='PC3',
                    title='PCA 3D Scatter Plot',
                    labels={'PC1': 'Principal Component 1',
                            'PC2': 'Principal Component 2',
                            'PC3': 'Principal Component 3'})

fig.write_html('/app/pca.html')

## K-Means

In [ ]:
scores_pca = pca.transform(nuclei_std)
scores_pca

In [ ]:
wcss = []
for i in range (1,21):
    kmeans_pca = KMeans(n_clusters = i, init = 'k-means++', random_state = 42)
    kmeans_pca.fit(scores_pca)
    wcss.append(kmeans_pca.inertia_)

In [ ]:
plt.figure()

plt.plot(wcss, marker = 'o', color='black')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.title('K-means with PCA')

# Mostrar el gráfico
plt.show()

In [ ]:
kmeans_pca = KMeans(n_clusters = 3, init = 'k-means++', random_state = 42)

In [ ]:
kmeans_pca.fit(scores_pca)

In [ ]:
df_segm_pca_kmeans = pd.concat([nuclei.reset_index(drop = True), pd.DataFrame(scores_pca)], axis = 1)
df_segm_pca_kmeans.columns.values[-3: ] = ['C1', 'C2', 'C3']
df_segm_pca_kmeans['Segment K-means PCA'] = kmeans_pca.labels_

In [ ]:
df_segm_pca_kmeans['Segment'] = df_segm_pca_kmeans['Segment K-means PCA'].map({
    0:'Primero',
    1:'Segundo',
    2:'Tercero'})

In [ ]:
df_plotly = df_segm_pca_kmeans.copy()
df_plotly['Segment K-means PCA'] = df_plotly['Segment K-means PCA'].map({
    0: 'Cluster 1',
    1: 'Cluster 2',
    2: 'Cluster 3'
})

fig = px.scatter_3d(df_plotly, x='C1', y='C2', z='C3', color='Segment K-means PCA',
                    labels={'C1': 'Componente 1', 'C2': 'Componente 2', 'C3': 'Componente 3'},
                    title='PCA with K-Means Clustering')

fig.write_html('/app/pca_kmeans.html')

In [ ]:
import numpy as np

# Obtener los centros de los clústeres
centers = kmeans.cluster_centers_

# Crear un DataFrame para los centros
centers_df = pd.DataFrame(centers, columns=['PC1', 'PC2', 'PC3'])

# Añadir un identificador de clúster
centers_df['Cluster'] = np.arange(len(centers))

# Mostrar los centros de los clústeres
print(centers_df)

In [ ]:
# Añadir las etiquetas de clúster al DataFrame original
nuclei['Cluster'] = labels

# Contar la cantidad de puntos en cada clúster
cluster_counts = nuclei['Cluster'].value_counts()
print(cluster_counts)

In [ ]:
# Calcular la puntuación de la silueta
silhouette_avg = silhouette_score(pca_result, labels)
print(f'Silhouette Score: {silhouette_avg}')

In [ ]:
import plotly.graph_objs as go
import plotly.express as px
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

# Supongamos que ya has ajustado PCA y KMeans
# pca = PCA(n_components=3)
# pca_result = pca.fit_transform(nuclei_std)
# kmeans = KMeans(n_clusters=3)
# labels = kmeans.fit_predict(pca_result)
# centers = kmeans.cluster_centers_

# Crear DataFrame de PCA con etiquetas de clúster
df_plotly = df_segm_pca_kmeans.copy()
df_plotly['Segment K-means PCA'] = df_plotly['Segment K-means PCA'].map({
    0: 'Cluster 1',
    1: 'Cluster 2',
    2: 'Cluster 3'
})

# Crear DataFrame para los centroides
centers_df = pd.DataFrame(centers, columns=['C1', 'C2', 'C3'])
centers_df['Cluster'] = ['Cluster 1', 'Cluster 2', 'Cluster 3']

# Crear el gráfico 3D
fig = px.scatter_3d(df_plotly, x='C1', y='C2', z='C3', color='Segment K-means PCA',
                    labels={'C1': 'Componente 1', 'C2': 'Componente 2', 'C3': 'Componente 3'},
                    title='PCA with K-Means Clustering')

# Añadir los centroides al gráfico
fig.add_trace(go.Scatter3d(
    x=centers_df['C1'],
    y=centers_df['C2'],
    z=centers_df['C3'],
    mode='markers',
    marker=dict(size=10, color='red', symbol='cross'),
    name='Centroids'
))

# Añadir esferas simuladas alrededor de los centroides
# Usar la función `scatter_3d` para dibujar esferas "blobs" alrededor de los centroides
for i, row in centers_df.iterrows():
    # Crear un conjunto de puntos para simular una esfera alrededor del centroide
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x = row['C1'] + 0.5 * np.outer(np.cos(u), np.sin(v))
    y = row['C2'] + 0.5 * np.outer(np.sin(u), np.sin(v))
    z = row['C3'] + 0.5 * np.outer(np.ones(np.size(u)), np.cos(v))
    
    fig.add_trace(go.Scatter3d(
        x=x.flatten(),
        y=y.flatten(),
        z=z.flatten(),
        mode='markers',
        marker=dict(size=3, color='red', opacity=0.2),  # Ajusta el tamaño y la opacidad para simular blobs
        name=f'Cluster {i+1} Blob'
    ))

# Actualizar el layout
fig.update_layout(
    legend_title='Cluster',
    margin=dict(l=0, r=0, t=40, b=0)
)

# Guardar el gráfico como un archivo HTML
fig.write_html('/app/pca_kmeans_blobs.html')

## UMap

In [ ]:
import umap
import plotly.express as px
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [ ]:
# Aplicar UMAP
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean', n_components=3)
umap_result = umap_model.fit_transform(nuclei_std)

In [ ]:
# Crear DataFrame para UMAP
df_umap = pd.DataFrame(umap_result, columns=['UMAP1', 'UMAP2', 'UMAP3'])
# Si tienes etiquetas o clústeres, agrégales aquí, por ejemplo:
# df_umap['Cluster'] = labels  # si ya tienes etiquetas de clústeres

In [ ]:
# Crear gráfico 3D si quieres explorar en 3 dimensiones (necesitas n_components=3)
umap_model_3d = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean', n_components=3)
umap_result_3d = umap_model_3d.fit_transform(nuclei_std)
df_umap_3d = pd.DataFrame(umap_result_3d, columns=['UMAP1', 'UMAP2', 'UMAP3'])

fig_3d = px.scatter_3d(df_umap_3d, x='UMAP1', y='UMAP2', z='UMAP3',
                      labels={'UMAP1': 'UMAP Dimension 1', 'UMAP2': 'UMAP Dimension 2', 'UMAP3': 'UMAP Dimension 3'},
                      title='UMAP Projection of Original Data (3D)')

fig_3d.write_html('/app/umap_3d.html')

## K-means

In [ ]:
# Aplicar K-Means
kmeans = KMeans(n_clusters=3)  # Ajusta el número de clústeres según sea necesario
clusters = kmeans.fit_predict(umap_result_3d)

# Agregar las etiquetas de clúster al DataFrame
df_umap_3d['Cluster'] = clusters

# Visualizar con Plotly
import plotly.express as px
fig = px.scatter_3d(df_umap_3d, x='UMAP1', y='UMAP2', z='UMAP3', color='Cluster',
                    title='UMAP with K-Means Clustering')
fig.write_html('/app/umap_kmeans.html')
